In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

sns.set(style="whitegrid")

### Data Loading

In [ ]:
df = pd.read_csv("StudentsPerformance.csv")
df.head()

### Shape of the dataset

In [ ]:
#Shape of the dataset (rows,columns)
print("Shape of dataset:", df.shape)

In [ ]:
df.info()

### Check Missing Values

In [ ]:
df.isnull().sum()

This means there are no missing values in the data set!

Check Duplicates

In [ ]:
df.duplicated().sum()

There are no duplicates values in the data set

### Checking the Number of Unique Values of Each Column

In [ ]:
df.nunique()

### Checking counts of unique values, for example in the gender feature

In [ ]:
df['gender'].value_counts()

### Check Statistics of Dataset

In [ ]:
df.describe(include='all').T


   

*   From above description of numerical data, all means are very close to each other - between 66 and 68.05;
*   All standard deviations are also close - between 14.6 and 15.19;
*   While there is a minimum score 0 for math, for writing minimum is much higher = 10 and for reading myet higher = 17


In [ ]:
#Check categories in each columns
print("Categories in 'gender' variable:     ", end=" ")
print(df['gender'].unique())

print("Categories in 'race/ethnicity' variable:  ", end=" ")
print(df['race/ethnicity'].unique())

print("Categories in 'parental level of education' variable:", end=" ")
print(df['parental level of education'].unique())

print("Categories in 'lunch' variable:     ", end=" ")
print(df['lunch'].unique())

print("Categories in 'test preparation course' variable:     ", end=" ")
print(df['test preparation course'].unique())

In [ ]:
# define numerical & categorical columns
numeric_features = [feature for feature in df.columns if df[feature].dtype != 'O']
categorical_features = [feature for feature in df.columns if df[feature].dtype == 'O']

# print columns
print('We have {} numerical features : {}'.format(len(numeric_features), numeric_features))
print('\nWe have {} categorical features : {}'.format(len(categorical_features), categorical_features))

In [ ]:
#  Try both here

# df['math score'].dtype
# df['race/ethnicity'].dtype

### Adding Columns for Total Score and Average

In [ ]:
df["total_score"] = df['math score'] + df["reading score"] + df["writing score"]
df['average_score'] = df['total_score'] / 3

df.head()


#### We'll classify whether a student passes math (score >= 60)

In [ ]:
def pass_or_fail(x):
    if x >= 50:
        return 1    # Pass
    else:
        return 0    # Fail


df['pass'] = df['average_score'].apply(pass_or_fail)

Another way to apply the lambda function, as follows:

In [ ]:
# We'll classify whether a student passes math (score >= 50)
#df['pass'] = df['average_score'].apply(lambda x: 1 if x >= 50 else 0)

In [ ]:
# Let's have a look on the Dataframe now
df

In [ ]:
df[['average_score', 'pass']].head(10)


## Visualize Some Relationships


In [ ]:
# Create a subplot with two histograms side by side
fig, axs = plt.subplots(1, 2, figsize=(15, 7))

# Plot the first histogram with KDE for the average score
plt.subplot(121)
sns.histplot(data=df, x='average_score', bins=30, kde=True, color='g')
plt.title('Distribution of Average Scores')
plt.xlabel('Average Score')
plt.ylabel('Frequency')

# Plot the second histogram with KDE, differentiated by gender
plt.subplot(122)
sns.histplot(data=df, x='average_score', kde=True, hue='gender')
plt.title('Distribution of Average Scores by Gender')
plt.xlabel('Average Score')
plt.ylabel('Frequency')

# Display the subplots
plt.tight_layout()
plt.show()

In general, Female students tend to perform well then male students.


In [ ]:
# Plot the first histogram with KDE, differentiated by lunch type (Overall)
sns.histplot(data=df, x='average_score', kde=True, hue='lunch')
plt.title('Distribution of Average Scores by Lunch (Overall)')
plt.xlabel('Average Score')
plt.ylabel('Frequency')

Standard lunch helps perform well in exams.

In [ ]:
plt.figure(figsize=(18,8))

plt.subplot(1, 4, 1)
plt.title('MATH SCORES')
sns.violinplot(y='math score',data=df,color='red',linewidth=3)

plt.subplot(1, 4, 2)
plt.title('READING SCORES')
sns.violinplot(y='reading score',data=df,color='green',linewidth=3)

plt.subplot(1, 4, 3)
plt.title('WRITING SCORES')
sns.violinplot(y='writing score',data=df,color='blue',linewidth=3)

plt.show()

* From the above three plots its clearly visible that most of the students score in between 60-80 in Maths whereas in reading and writing most of them score from 50-80

#### CHECKING OUTLIERS

In [ ]:
plt.subplots(1,4,figsize=(16,5))
plt.subplot(141)
sns.boxplot(df['math score'],color='skyblue')
plt.subplot(142)
sns.boxplot(df['reading score'],color='hotpink')
plt.subplot(143)
sns.boxplot(df['writing score'],color='yellow')
plt.subplot(144)
sns.boxplot(df['average_score'],color='lightgreen')
plt.show()

In [ ]:
# ==============================================
# Preprocess Features
# ==============================================

# Select relevant features
X = df[[ 'gender', 'race/ethnicity',
        'parental level of education',
         'lunch', 'test preparation course']]

#Target Label
y = df['pass']


In [ ]:
# Let's have a look on the Trainig dataframe (X)
X

In [ ]:
#Let's have a look on the Trainig dataframe (y)
y

## Hot-Encode Categorical Data
Convert categorical features into numerical format using one-hot encoding so each category is represented by binary (0/1) columns.

We do one-hot encoding because most machine learning algorithms cannot work directly with categorical (text) data. By converting categories into numeric binary columns, the model can understand and process the data correctly without assuming any order or hierarchy between categories.

In [ ]:
# Hot-Encode categorical data
X = pd.get_dummies(X)


#### Label Encoding


In [ ]:
# from sklearn.preprocessing import LabelEncoder
# lc = LabelEncoder()
# X['gender'] = lc.fit_transform(df['gender'])
# X['race/ethnicity'] = lc.fit_transform(df['race/ethnicity'])
# X['parental level of education'] = lc.fit_transform(df['parental level of education'])
# X['lunch'] = lc.fit_transform(df['lunch'])
# X['test preparation course'] = lc.fit_transform(df['test preparation course'])
# X.head()

Now, let's have a look on the Traingin Data frame ```X``` after the one hot encoding

In [ ]:
X

## Train-Test Split

Divide the dataset into training and testing sets.

* The training set is used to fit the model
* The testing set is used to evaluate its performance on unseen data, helping to check for overfitting and generalization.

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

Now, check the size of the traing set and test set

In [ ]:
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

# Applying the Naive Bayes ML model

Probabilistic classifier assuming feature independence. GaussianNB for continuous features.


In [ ]:
# Naive Bayes
naive_bayes_model = GaussianNB()

### Train Logistic Regression


In [ ]:
# Train Logistic Regression
naive_bayes_model.fit(X_train, y_train)

# Evaluate
naive_bayes_accuracy = naive_bayes_model.score(X_test, y_test)
print("Naiive Bayes model Accuracy:", naive_bayes_accuracy)

# Applying Decision Tree ML model
DT model splits data into branches based on features.

In [ ]:
# Decision Tree
# Splits data into branches based on features. max_depth=4 prevents overfitting.
decision_tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)

In [ ]:
# Train Decision Tree
decision_tree_model.fit(X_train, y_train)

# Evaluate Decision Tree
dt_accuracy = decision_tree_model.score(X_test, y_test)
print("Decision Tree Accuracy:", dt_accuracy)

# ///////////////////////////////Multiple Models/////////////////////////////////

In [ ]:
# ==============================================
# Define and Train Multiple Classifiers
# ==============================================

models = {
    "Logistic Regression": LogisticRegression(),

    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),

    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),

    "KNN": KNeighborsClassifier(n_neighbors=5),

    "Naive Bayes": GaussianNB(),

    "SVM": SVC(kernel='rbf', probability=True, random_state=42)
}

In [ ]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    results.append((name, acc))
    print(f"{name} Accuracy: {acc:.4f}")

results_df = pd.DataFrame(results, columns=["Model", "Accuracy"]).sort_values(by="Accuracy", ascending=False)


In [ ]:
# ==============================================
# 8️⃣ Compare Model Performance
# ==============================================

plt.figure(figsize=(8,5))
sns.barplot(x='Accuracy', y='Model', data=results_df, palette="viridis")
plt.title("Model Accuracy Comparison")
plt.xlim(0, 1)
plt.show()

results_df

In [ ]:
# ==============================================
# 9️⃣ Confusion Matrix & Classification Report
# ==============================================

# Pick the best-performing model
best_model_name = results_df.iloc[0, 0]
best_model = models[best_model_name]

y_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Fail', 'Pass'], yticklabels=['Fail', 'Pass'])
plt.title(f"Confusion Matrix - {best_model_name}")
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

print(f"🔹 Classification Report for {best_model_name}:\n")
print(classification_report(y_test, y_pred))


# Home Work Task 

Regression Task: Diabetes Dataset (Scikit-Learn)

### Objective
Build and evaluate a regression model to predict disease progression using the Diabetes dataset from `scikit-learn`.

### Instructions

1. **Load and Explore the Dataset**
   - Use the `load_diabetes()` function from `sklearn.datasets`. use this code to Load the dataset into a DataFrame
   ```python
   from sklearn.datasets import load_diabetes
   import pandas as pd

   data = load_diabetes()
   df = pd.DataFrame(data.data, columns=data.feature_names)
   df['target'] = data.target
   ```
   - Show the first 5 rows in the dataset
   - Inspect the shape of the dataset (rows, columns)
   - Inspect the data (features, target, summary statistics).

2. **Prepare the Data**
   - Separate features and target.
   - Split the data into training and testing sets (80/20).

3. **Build a Regression Model**
   - Choose an appropriate regression algorithm (e.g., Linear Regression).
   - Train the model on the training data.
     
4. **Evaluate the Model**
   - Predict on the test set.

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load and inspect the dataset
data = load_diabetes()
df_diabetes = pd.DataFrame(data.data, columns=data.feature_names)
df_diabetes["target"] = data.target

print("First five rows:")
display(df_diabetes.head())
print("Dataset shape:", df_diabetes.shape)
print("\nDataset information:")
df_diabetes.info()
print("\nSummary statistics:")
display(df_diabetes.describe())

# Separate the predictors from the disease-progression target
X_diabetes = df_diabetes.drop(columns="target")
y_diabetes = df_diabetes["target"]

# Split the data into training and testing sets (80/20)
X_train_diabetes, X_test_diabetes, y_train_diabetes, y_test_diabetes = train_test_split(
    X_diabetes, y_diabetes, test_size=0.2, random_state=42
)

# Train a linear regression model
regression_model = LinearRegression()
regression_model.fit(X_train_diabetes, y_train_diabetes)

# Predict on the test set and evaluate the model
y_pred_diabetes = regression_model.predict(X_test_diabetes)
mae = mean_absolute_error(y_test_diabetes, y_pred_diabetes)
rmse = np.sqrt(mean_squared_error(y_test_diabetes, y_pred_diabetes))
r2 = r2_score(y_test_diabetes, y_pred_diabetes)

print(f"\nTraining samples: {len(X_train_diabetes)}")
print(f"Testing samples: {len(X_test_diabetes)}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R-squared (R^2): {r2:.3f}")

# Compare actual and predicted target values for the first five test examples
comparison = pd.DataFrame({
    "Actual": y_test_diabetes,
    "Predicted": y_pred_diabetes
})
display(comparison.head())

First five rows:


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


Dataset shape: (442, 11)

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 442 entries, 0 to 441
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     442 non-null    float64
 1   sex     442 non-null    float64
 2   bmi     442 non-null    float64
 3   bp      442 non-null    float64
 4   s1      442 non-null    float64
 5   s2      442 non-null    float64
 6   s3      442 non-null    float64
 7   s4      442 non-null    float64
 8   s5      442 non-null    float64
 9   s6      442 non-null    float64
 10  target  442 non-null    float64
dtypes: float64(11)
memory usage: 38.1 KB

Summary statistics:


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
count,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,442.000000
mean,-2.511817e-19,1.230790e-17,-2.245564e-16,-4.797570e-17,-1.381499e-17,3.918434e-17,-5.777179e-18,-9.042540e-18,9.293722e-17,1.130318e-17,152.133484
std,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,77.093005
min,-1.072256e-01,-4.464164e-02,-9.027530e-02,-1.123988e-01,-1.267807e-01,-1.156131e-01,-1.023071e-01,-7.639450e-02,-1.260971e-01,-1.377672e-01,25.000000
25%,-3.729927e-02,-4.464164e-02,-3.422907e-02,-3.665608e-02,-3.424784e-02,-3.035840e-02,-3.511716e-02,-3.949338e-02,-3.324559e-02,-3.317903e-02,87.000000
50%,5.383060e-03,-4.464164e-02,-7.283766e-03,-5.670422e-03,-4.320866e-03,-3.819065e-03,-6.584468e-03,-2.592262e-03,-1.947171e-03,-1.077698e-03,140.500000
75%,3.807591e-02,5.068012e-02,3.124802e-02,3.564379e-02,2.835801e-02,2.984439e-02,2.931150e-02,3.430886e-02,3.243232e-02,2.791705e-02,211.500000
max,1.107267e-01,5.068012e-02,1.705552e-01,1.320436e-01,1.539137e-01,1.987880e-01,1.811791e-01,1.852344e-01,1.335973e-01,1.356118e-01,346.000000



Training samples: 353
Testing samples: 89
Mean Absolute Error (MAE): 42.79
Root Mean Squared Error (RMSE): 53.85
R-squared (R^2): 0.453


,Actual,Predicted
287,219.0,139.547558
211,70.0,179.517208
72,202.0,134.038756
321,230.0,291.417029
73,111.0,123.789659
